<a href="https://colab.research.google.com/github/fralfaro/ics294-latex/blob/main/colab/certamen1_ejemplos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Certamen 1: ejemplos en R

Notebook con ejemplos sencillos para repasar las semanas 1, 2, 3 y 4 de Econometría ICS-294.

Para ejecutar código en **R** dentro de Google Colab, vaya a **Runtime > Change runtime type** y seleccione **R** como lenguaje.


## Preparación

Usaremos `dplyr` para manipular datos y `ggplot2` para gráficos. En Google Colab con kernel de R asumimos que estos paquetes están disponibles.

In [ ]:
library(dplyr)
library(ggplot2)

set.seed(294)
options(scipen = 999)

tema_curso <- theme_minimal()
tema_curso

## Semana 1: datos, estadística descriptiva y correlación

Idea: antes de estimar modelos, miramos los datos. Calculamos promedios, dispersión, correlaciones y gráficos simples.

In [ ]:
n <- 80

datos <- tibble(
  educ = sample(8:18, n, replace = TRUE),
  exp = sample(0:30, n, replace = TRUE),
  salario = 300 + 55 * educ + 12 * exp + rnorm(n, mean = 0, sd = 120)
)

datos %>%
  slice_head(n = 6)

In [ ]:
datos %>%
  summarise(
    n = n(),
    salario_promedio = mean(salario),
    salario_mediano = median(salario),
    desviacion_salario = sd(salario),
    educ_promedio = mean(educ),
    exp_promedio = mean(exp),
    cor_educ_salario = cor(educ, salario)
  )

datos %>%
  group_by(educ) %>%
  summarise(
    salario_promedio = mean(salario),
    n = n(),
    .groups = "drop"
  ) %>%
  arrange(educ)

In [ ]:
ggplot(datos, aes(x = salario)) +
  geom_histogram(bins = 18, fill = "gray80", color = "white") +
  labs(
    title = "Distribución de salarios",
    x = "Salario",
    y = "Frecuencia"
  ) +
  theme_minimal()

ggplot(datos, aes(x = educ, y = salario)) +
  geom_point(color = "steelblue", size = 2, alpha = 0.8) +
  labs(
    title = "Relación entre educación y salario",
    x = "Años de educación",
    y = "Salario"
  ) +
  theme_minimal()

## Semana 2: regresión lineal simple

Modelo poblacional: $y = \beta_0 + \beta_1 x + u$.

En el ejemplo estimamos cómo cambia el salario promedio cuando aumenta la educación.

In [ ]:
modelo_simple <- lm(salario ~ educ, data = datos)

summary(modelo_simple)

In [ ]:
coeficientes_simple <- tibble(
  termino = names(coef(modelo_simple)),
  estimacion = as.numeric(coef(modelo_simple))
)

coeficientes_simple

coeficientes_simple %>%
  filter(termino == "educ") %>%
  mutate(
    interpretacion = paste0(
      "Un año adicional de educación se asocia con ",
      round(estimacion, 2),
      " unidades más de salario en promedio."
    )
  )

In [ ]:
ggplot(datos, aes(x = educ, y = salario)) +
  geom_point(color = "steelblue", size = 2, alpha = 0.8) +
  geom_smooth(method = "lm", se = FALSE, color = "firebrick", linewidth = 1) +
  labs(
    title = "Regresión lineal simple",
    x = "Años de educación",
    y = "Salario"
  ) +
  theme_minimal()

In [ ]:
datos_modelo_simple <- datos %>%
  mutate(
    salario_estimado = fitted(modelo_simple),
    residuo = resid(modelo_simple)
  )

datos_modelo_simple %>%
  select(salario, salario_estimado, residuo) %>%
  slice_head(n = 6)

datos_modelo_simple %>%
  summarise(suma_residuos = sum(residuo))

## Semana 3: causalidad, tratamiento y sesgo de selección

Una diferencia de promedios puede confundir efecto causal con diferencias previas entre grupos. Aquí simulamos un tratamiento y comparamos un caso aleatorio con un caso con selección.

In [ ]:
n <- 200

datos_exp <- tibble(
  habilidad = rnorm(n),
  tratamiento = rbinom(n, size = 1, prob = 0.5),
  y0 = 500 + 80 * habilidad + rnorm(n, 0, 60),
  efecto_real = 100,
  y = y0 + efecto_real * tratamiento
 )

datos_exp %>%
  group_by(tratamiento) %>%
  summarise(
    promedio_y = mean(y),
    promedio_habilidad = mean(habilidad),
    n = n(),
    .groups = "drop"
  )

datos_exp %>%
  summarise(
    diferencia_promedios = mean(y[tratamiento == 1]) - mean(y[tratamiento == 0]),
    efecto_real = first(efecto_real)
  )

In [ ]:
datos_sel <- datos_exp %>%
  transmute(
    habilidad,
    tratamiento = if_else(habilidad > median(habilidad), 1, 0),
    y0,
    efecto_real,
    y = y0 + efecto_real * tratamiento
  )

datos_sel %>%
  group_by(tratamiento) %>%
  summarise(
    promedio_y = mean(y),
    promedio_y0 = mean(y0),
    promedio_habilidad = mean(habilidad),
    n = n(),
    .groups = "drop"
  )

datos_sel %>%
  summarise(
    diferencia_observada = mean(y[tratamiento == 1]) - mean(y[tratamiento == 0]),
    efecto_real = first(efecto_real),
    sesgo_seleccion = mean(y0[tratamiento == 1]) - mean(y0[tratamiento == 0])
  )

In [ ]:
datos_sel %>%
  mutate(tratamiento = factor(tratamiento, labels = c("Control", "Tratado"))) %>%
  ggplot(aes(x = tratamiento, y = y, fill = tratamiento)) +
  geom_boxplot(alpha = 0.7, width = 0.55) +
  labs(
    title = "Resultados observados por grupo",
    x = "Grupo",
    y = "Resultado observado"
  ) +
  guides(fill = "none") +
  theme_minimal()

En el caso con selección, los tratados ya tenían mayor resultado potencial sin tratamiento. Por eso la diferencia observada no corresponde necesariamente al efecto causal.

## Semana 4: regresión múltiple y controles

Modelo: $y = \beta_0 + \beta_1x_1 + \beta_2x_2 + u$.

La interpretación de $\beta_1$ es ceteris paribus: cambio promedio en $y$ cuando cambia $x_1$, manteniendo constantes las otras variables incluidas.

In [ ]:
modelo_multiple <- lm(salario ~ educ + exp, data = datos)

summary(modelo_simple)
summary(modelo_multiple)

In [ ]:
comparacion <- tibble(
  modelo = c("Simple: salario ~ educ", "Múltiple: salario ~ educ + exp"),
  beta_educ = c(coef(modelo_simple)[["educ"]], coef(modelo_multiple)[["educ"]]),
  r2 = c(summary(modelo_simple)$r.squared, summary(modelo_multiple)$r.squared)
)

comparacion

datos %>%
  mutate(
    salario_estimado_multiple = fitted(modelo_multiple),
    residuo_multiple = resid(modelo_multiple)
  ) %>%
  select(salario, educ, exp, salario_estimado_multiple, residuo_multiple) %>%
  slice_head(n = 6)

In [ ]:
datos %>%
  mutate(salario_estimado_multiple = fitted(modelo_multiple)) %>%
  ggplot(aes(x = salario_estimado_multiple, y = salario)) +
  geom_point(color = "steelblue", size = 2, alpha = 0.8) +
  geom_abline(intercept = 0, slope = 1, color = "firebrick", linewidth = 1) +
  labs(
    title = "Salario observado versus salario estimado",
    x = "Salario estimado",
    y = "Salario observado"
  ) +
  theme_minimal()

In [ ]:
X <- model.matrix(modelo_multiple)
y <- datos$salario

beta_matriz <- solve(t(X) %*% X) %*% t(X) %*% y

beta_matriz
coef(modelo_multiple)

## Espacio para próximas semanas

Las siguientes secciones quedan reservadas para agregar más ejemplos cuando se necesiten.

## Semana 5: pendiente

_Agregar ejemplo aquí._

In [ ]:
# Código semana 5


## Semana 6: pendiente

_Agregar ejemplo aquí._

In [ ]:
# Código semana 6


## Semana 7: pendiente

_Agregar ejemplo aquí._

In [ ]:
# Código semana 7


## Semana 8: pendiente

_Agregar ejemplo aquí._

In [ ]:
# Código semana 8
